# 4.2 Functions Builtins

**Prerequisites:** 4.1 Functions User-defined
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
Builtins grouped **by the job they do**:
- **Introspection** — `dir`, `help`, `__doc__`, `isinstance`, `callable`, `getattr`/`setattr`/`hasattr`, `vars`, `object`
- **Iteration tools** — `enumerate`, `zip`, `reversed`, `sorted` (+ **`key=` functions**)
- **Aggregation & truth** — `len`, `sum`, `min`, `max`, `any`, `all`, `abs`, `divmod`, `pow`, `round`
- **Conversion & construction** — `int`, `float`, `str`, `bool`, `list`, `tuple`, `set`, `dict`
- **Functional tools** — `lambda`, `map`, `filter`, `functools.reduce` — and when a comprehension is better
- Forward pointers: `classmethod`, `staticmethod`, `property` (OOP)
- A compact reference table at the end

---

## Why builtins first?

Python ships with roughly 70 functions that are **always available — no import**. They are the language's working vocabulary: most are implemented in C (fast), and more importantly they are *idiomatic* — a reader instantly understands `any(...)` where a hand-rolled flag-and-loop needs studying.

This notebook groups them **by the job they do**, because that is how you reach for them:

| Group | Question it answers | Tools |
|---|---|---|
| Introspection | "What is this object? What can it do?" | `dir`, `help`, `isinstance`, `getattr`, ... |
| Iteration | "Loop over this — without index bookkeeping" | `enumerate`, `zip`, `reversed`, `sorted` |
| Aggregation & truth | "Collapse this collection to one fact" | `len`, `sum`, `min`/`max`, `any`/`all` |
| Conversion | "Turn this into that type" | `int`, `str`, `list`, `dict`, ... |
| Functional | "Apply / keep / combine across a collection" | `lambda`, `map`, `filter`, `reduce` |

The full list is one `dir(__builtins__)` away — which is the first stop below.

---

## Group 1 — Introspection: asking objects about themselves

**When you reach for these:** debugging in a REPL ("what methods does this thing have?"), writing generic code that must work on objects it has never seen, and building plugin-style systems that look behaviour up **by name** instead of hard-coding it.

### dir(): Returns an object’s attributes.
- dir() works on ANY object — a module, class, instance, function or builtin — and returns a sorted list of its attribute names.
- Called with no arguments, dir() lists the names in the current local scope.
- For a module, the list contains all the sub-modules, variables and functions defined in that module.

In [ ]:
print(dir(__builtins__))

### `help()` — the interactive documentation browser
`help(obj)` prints documentation for any module, function, class, keyword or topic. Under the hood it reads the **docstrings** (`__doc__`) you learned to write in **4.1** — which is the practical reason to write them.

In [ ]:
# help() blocks waiting for interaction, so it stays commented here.
# Uncomment in a live session to explore:
# help('modules')      # every importable module
# help(len)            # docs for one object
# help('keywords')     # the language keywords
print("help() displays the same __doc__ strings we read directly below.")

#### `__doc__` — the docstring attribute
Every function (builtin or yours) carries its docstring as `.__doc__` — this is exactly what `help()` displays.

In [ ]:
print(abs.__doc__)

### `isinstance(obj, type)` — type checking that respects inheritance
Returns `True` if `obj` is an instance of `type` **or of any subclass** — which is why it beats comparing `type(x) == T`. It also accepts a tuple of types, meaning "any of these".

In [ ]:
a = 5
print(isinstance(a, int))

days = [1, 2, 3, 4, 5, 6, 7]
print(isinstance(days, list))

# a tuple of types = "any of these"
print(isinstance(3.5, (int, float)))

### `callable`, `getattr` and friends — programming *with* names

These let a program treat its own structure as data: test whether something is callable, and read/write attributes whose names arrive at run time (from a config file, an API payload, a CLI flag).

| Builtin | Does |
|---|---|
| `callable(x)` | Can `x` be called? |
| `getattr(obj, name[, default])` | Read an attribute **by name** — the safe alternative to `eval` |
| `setattr` / `hasattr` / `delattr` | Write / test / remove, likewise |
| `vars(obj)` | The object's `__dict__` |
| `isinstance(x, T)` | Type test that respects inheritance |

⚠️ `isinstance(True, int)` is `True`, because `bool` subclasses `int` (see **2.2**).

In [ ]:
# ---- callable(): can this object be called? ----
def f(): pass

print("callable(f)      :", callable(f))
print("callable(len)    :", callable(len))
print("callable('abc')  :", callable("abc"))
print("callable(str)    :", callable(str), " <- classes are callable; calling one makes an instance")


# ---- getattr / setattr / hasattr: attribute access by NAME ----
class Config:
    host = "localhost"
    port = 8080

cfg = Config()

print("\ngetattr(cfg, 'host')          :", getattr(cfg, "host"))
print("getattr(cfg, 'missing', 'n/a'):", getattr(cfg, "missing", "n/a"))
print("hasattr(cfg, 'port')          :", hasattr(cfg, "port"))

setattr(cfg, "debug", True)
print("after setattr                 :", cfg.debug)

# Why this matters: driving behaviour from data, without eval()
for name in ["host", "port", "debug"]:
    print(f"  {name:<6} = {getattr(cfg, name)}")


# ---- vars() and dir() ----
print("\nvars(cfg):", vars(cfg), " <- the instance __dict__")
print("public attrs:", [a for a in dir(cfg) if not a.startswith("_")])


# ---- isinstance vs type ----
print("\nisinstance(True, int):", isinstance(True, int), " <- bool subclasses int!")
print("type(True) is int    :", type(True) is int)

def is_real_int(x):
    return isinstance(x, int) and not isinstance(x, bool)

print("is_real_int(5)   :", is_real_int(5))
print("is_real_int(True):", is_real_int(True))

# isinstance accepts a tuple of types
print("\nisinstance(3.5, (int, float)):", isinstance(3.5, (int, float)))

### `object()` — the featureless base
`object` is the root class of everything in Python. A bare `object()` has no settable attributes — its main practical use is as a unique **sentinel**: a value guaranteed not to collide with any real data (not even `None`).

In [ ]:
o = object()
print(type(o))

# Practical use: a sentinel distinguishable from every legitimate value
MISSING = object()

def get_setting(store, key, default=MISSING):
    value = store.get(key, MISSING)
    if value is MISSING:
        if default is MISSING:
            raise KeyError(key)
        return default
    return value

print(get_setting({'retries': 0}, 'retries'))    # 0 is a real stored value, not "missing"
print(get_setting({}, 'retries', default=3))     # falls back to the default

In [ ]:
# The group in action — dispatch incoming job names to handler methods, safely
class JobHandlers:
    def handle_backup(self):
        return 'backup started'

    def handle_reindex(self):
        return 'reindex started'

incoming = ['backup', 'reindex', 'shutdown']
handlers = JobHandlers()
for job in incoming:
    fn = getattr(handlers, f'handle_{job}', None)   # look the method up BY NAME
    if callable(fn):
        print(f'{job:<9} -> {fn()}')
    else:
        print(f'{job:<9} -> no handler registered, skipped')

---

## Group 2 — Iteration tools: loop without bookkeeping

Most day-job Python is a loop over a collection. These builtins remove the error-prone index arithmetic: `enumerate` numbers items for you, `zip` walks parallel sequences together, `reversed` walks backwards, and `sorted` (with `key=`) orders by any criterion.

⚠️ All but `sorted` return **lazy iterators** — values are produced on demand and can be consumed **once**. (`range`, the fifth member of this family, was covered in **03 Flow Control**.)

### `enumerate(iterable, start=0)` — a counter for free
Yields `(index, item)` pairs, lazily. **Why:** whenever you need the position *and* the value — numbering log lines, reporting *which* row failed validation — `enumerate` replaces the manual `i = 0; ...; i += 1` dance.

In [ ]:
lines = ['INFO  boot complete', 'WARN  disk 81%', 'ERROR raid degraded']

s = enumerate(lines)
print(s)              # an enumerate object — lazy, like map and zip
print(type(s))
print(list(s))        # materialised: (index, item) pairs

In [ ]:
# start= sets the first counter value — line numbers start at 1
for no, line in enumerate(lines, start=1):
    print(f'{no:>3}: {line}')

In [ ]:
# Keep every second reading from a sensor stream (indices 0, 2, 4, ...)
readings = [10, 11, 19, 14, 6, 4, 23]
sampled = []
for i, r in enumerate(readings):
    if i % 2 == 0:
        sampled.append(r)
print(sampled)

### `zip(*iterables)` — walk parallel sequences together
Combines multiple iterables element-wise: the *n*-th tuple holds the *n*-th item of each. **Why:** related data often arrives as parallel lists (the columns of a CSV, names + values); `zip` stitches the rows back together — or builds a `dict` from separate key/value sequences.
- Returns a lazy zip object; each `next()` yields one tuple.
- **Syntax:** `zip(iterable1, iterable2, ...)`

In [ ]:
hosts   = ['edge-01', 'edge-02', 'core-01']
status  = ['up', 'down', 'up']
latency = [12, 340, 48]

rows = zip(hosts, status, latency)
print(rows)
print(type(rows))
print(list(rows))                    # rows of a status table

# The classic move: two parallel sequences -> one dict
print(dict(zip(hosts, status)))

In [ ]:
# ⚠️ zip stops at the SHORTEST input — silently
print(list(zip([1, 2, 3], 'ab')))

# Python 3.10+: strict=True raises instead of truncating — use it when the
# lengths MUST match (e.g. columns of the same table)
try:
    list(zip([1, 2, 3], 'ab', strict=True))
except ValueError as exc:
    print('strict=True ->', exc)

In [ ]:
# zip(*zipped) UN-zips: rows back into columns
services = ['auth', 'billing', 'search', 'export']
errors   = [0, 3, 1, 7]
rows = zip(services, errors)

names, counts = zip(*rows)
print(list(names))
print(list(counts))

### reversed(): Returns a reverse iterator over a sequence.
- The argument must be a sequence (something with `__len__` and `__getitem__`, like list/tuple/str/range) or an object implementing `__reversed__` — not just any iterable (a set or generator won't do).
- It returns a new reverse iterator; the original object is NOT modified.

In [ ]:
deploys = ['v1.0', 'v1.1', 'v1.2']     # oldest -> newest
r = reversed(deploys)
print(r)                                # a lazy iterator; nothing is copied or modified
for version in r:                       # walk newest-first
    print(version)
print(deploys)                          # the original list is untouched

### `key=` functions

`sorted`, `max`, `min` (and `list.sort`) all accept a **`key` function**. It is called once
per element, and the comparison uses the *result* — so you sort by a computed value without
touching the data.

⚠️ `key` takes the function **itself**, uncalled: `key=len`, never `key=len()`.

In [ ]:
from operator import itemgetter, attrgetter

words = ["banana", "kiwi", "apple", "cherry"]
people = [("Aditya", 25), ("Priya", 31), ("Rahul", 19)]

# key= takes the FUNCTION ITSELF - no parentheses
print("by length     :", sorted(words, key=len))
print("longest       :", max(words, key=len))
print("shortest      :", min(words, key=len))

# itemgetter beats lambda for plain index/key access
print("\nby age        :", sorted(people, key=itemgetter(1)))
print("oldest        :", max(people, key=itemgetter(1)))

# Multi-key: return a tuple
data = [("b", 2), ("a", 2), ("c", 1)]
print("\nby value then name:", sorted(data, key=lambda t: (t[1], t[0])))

# attrgetter for objects
from types import SimpleNamespace
users = [SimpleNamespace(name="Z", age=20), SimpleNamespace(name="A", age=30)]
print("\nby .name      :", [u.name for u in sorted(users, key=attrgetter("name"))])

# sum() takes a start value
print("\nsum(start=100):", sum([1, 2, 3], 100))
print("flatten lists :", sum([[1, 2], [3, 4]], []))   # works, but slow - see itertools.chain

# math.prod is the `reduce(mul)` you actually want
import math
print("\nmath.prod     :", math.prod([1, 2, 3, 4, 5]))

# ⚠️ Common mistake: calling the key function
try:
    sorted(words, key=len())
except TypeError as exc:
    print("\nkey=len() ->", exc)

In [ ]:
# The group in action — a ranked latency report
hosts = ['edge-01', 'edge-02', 'core-01', 'core-02']
ms    = [48, 12, 31, 25]

report = sorted(zip(hosts, ms), key=lambda row: row[1])     # zip -> sort by latency
for rank, (host, latency) in enumerate(report, start=1):    # enumerate -> rank
    print(f'{rank}. {host:<8} {latency:>3} ms')

---

## Group 3 — Aggregation & truth: one fact from many values

These collapse a whole collection into a single answer: how many (`len`), the total (`sum`), the extremes (`min`/`max`), or a yes/no verdict (`any`/`all`). **Why:** they replace flag-and-loop boilerplate and read like the sentence you would say aloud — "if **all** checks passed", "if **any** host is down".

### `all()` and `any()` — collective truth
- `all(iterable)` → `True` if **every** element is truthy — a validation gate.
- `any(iterable)` → `True` if **at least one** element is truthy — an alert trigger.

Both **short-circuit**: `all` stops at the first falsy element, `any` at the first truthy one.

In [ ]:
lst1 = [False, True, True]
print(all(lst1))
print(any(lst1))

In [ ]:
lst2 = [1, 2, 3, 4]
print(all(lst2))
print(any(lst2))

In [ ]:
lst3 = [0, 2, 4, 6]  # 0 is present means false in boolean
print(all(lst3))
print(any(lst3))

In [ ]:
lst4 = []      # ⚠️ empty iterable: all([]) is True, but any([]) is False
# Why: all() is vacuously true — there is no element that could be falsy;
# any() needs at least one true element as a witness, and finds none.
print(all(lst4))
print(any(lst4))

### `abs`, `divmod`, `pow`, `round` — numeric helpers

- `abs(x)` — magnitude: how far from zero, sign ignored (the *size* of an error or drift).
- `divmod(a, b)` — `(a // b, a % b)` in one call: quotient **and** remainder together.
- `pow(x, y)`, and `pow(x, y, mod)` — power, and efficient *modular* power (the workhorse of public-key cryptography).
- `round(x, ndigits=None)` — ⚠️ banker's rounding: ties go to the **even** neighbour (see **2.2**). With the default `ndigits=None` it returns an `int`.

In [ ]:
drift = -3.7
print('abs   :', abs(drift), 'ms of clock drift, direction ignored')

minutes, seconds = divmod(754, 60)          # seconds -> minutes + seconds in one step
print('divmod:', f'754s = {minutes}m {seconds}s')
print('same  :', (754 // 60, 754 % 60))

print('pow   :', pow(5, 2))
print('modpow:', pow(5, 2, 3), ' <- (5**2) % 3, computed efficiently')

print('round :', round(2.5), round(3.5), ' <- ties go to the EVEN neighbour')

### `len`, `sum`, `min`, `max` — the everyday reducers
So common they hide in plain sight — and each one replaces a whole `reduce` call (see the functional group below).

In [ ]:
samples = [48, 12, 31, 25]          # response times, ms
print('count:', len(samples))
print('total:', sum(samples))
print('min  :', min(samples), '  max:', max(samples))
print('mean :', round(sum(samples) / len(samples), 1))

In [ ]:
# The group in action — an SLA health gate
latencies = {'edge-01': 48, 'edge-02': 12, 'core-01': 31}
SLA = 40

print('all within SLA :', all(ms <= SLA for ms in latencies.values()))
print('any breach     :', any(ms > SLA for ms in latencies.values()))
print('worst host     :', max(latencies, key=latencies.get),
      f'({max(latencies.values())} ms)')
print('breaching      :', [h for h, ms in latencies.items() if ms > SLA])

---

## Group 4 — Conversion & construction

`int`, `float`, `str`, `bool`, `list`, `tuple`, `set` and `dict` are builtins too — each is a *class*, and calling it converts or constructs (full treatment in **02 Datatypes**). Here, only the part that bites in real code: **everything arriving from outside — files, environment variables, HTTP, `input()` — is a string** until you convert it.

In [ ]:
# Config values arrive as strings
port = int('8080')
timeout = float('2.5')
print(port + 1, timeout * 2)

# ⚠️ bool() of a NON-EMPTY string is always True — including 'False' and '0'!
print(bool('False'), bool('0'), bool(''))
flag = 'False'
enabled = flag.lower() in ('1', 'true', 'yes')     # parse truthiness yourself
print('enabled:', enabled)

# Constructors convert between containers
print(list('abc'), tuple([1, 2]), set([1, 1, 2]), dict([('a', 1)]))

---

## Group 5 — Functional tools: functions as values

In Python a function is an object (**4.1**), so it can be *passed to* another function. That is the whole idea of this group: `map` applies a function to every element, `filter` keeps the elements a predicate approves, `reduce` folds a sequence into one value — and `lambda` writes the tiny throwaway functions they take.

Python's honest position, spelled out at the end of the group: **comprehensions usually beat `map`/`filter` with a `lambda`** — but `map(named_function, xs)` holds its own.

### `lambda` — anonymous, single-expression functions

`lambda arguments: expression` builds a function with **no name and exactly one expression**, whose value is the implicit return.

- Mathematically it is the arrow form: `f(x) = x² - x + 42` written as `x → x² - x + 42`.
- **When:** a function so small that naming it adds nothing — usually inline, as an argument to `sorted(key=...)`, `map()`, `filter()`.
- **When not:** the moment it needs a second expression, a statement, or a docstring — use `def`.

In [ ]:
# Program for addition of two no.
def func(x, y):
    return x+y
c = func(3,4)
print(c)

# Or using lambda()
func = lambda x,y : x+y
print(func(3,4))
print(func("Hello ", "Python"))

#### Tiny lambdas in the wild: predicates and key extractors

In [ ]:
is_even = lambda x: x % 2 == 0        # a predicate
last_char = lambda s: s[-1]           # a key extractor

print(is_even(8), is_even(7))
print(last_char('Python'))

# Their natural habitat is inline, as an argument:
print(sorted(['alpha', 'kiwi', 'box'], key=last_char))

#### ⚠️ Trap: a lambda reads outer names when it is *called*, not when it is defined
The next cell preserves a real bug from the original notes — worth reading closely.

In [ ]:
# ⚠️ The original read `f"{n} is ..."` inside the lambda — that interpolates the
# GLOBAL n, not the parameter x. It only worked because n happened to hold the
# same value that was passed in. Always use the parameter:
even = lambda x: f"{x} is Even" if (x % 2 == 0) else f"{x} is Odd"

# Interactive variant: n = int(input("Enter a number: "))
n = 7
print(even(n))

### map(): Executes a specified function for each item in a iterable sequence.
- **Syntax:** `map(function, iterable)`
    - function : It is a function to which map passes each element of given iterable.
    - iterable : It is a iterable which is to be mapped.
- map() return iterator map object.    
- **NOTE:** We can pass one or more iterable to the map() function.
<img src='./Image/4.2 Image a.png' width=60% height=60%/>

#### The canonical comparison: loop → `map(named function)` → `map(lambda)`
Same result three ways — watch how the intent gets denser each time.

In [ ]:
numbers = [1, 2, 3, 4]

# Normal method:
squared = []
for num in numbers:
    squared.append(num ** 2)
print(squared)

# Using map():
def square(num):
    return num ** 2

squared = list(map(square, numbers))
print(squared)

# Using map() function and lambda expression:
squared = list(map(lambda num: num**2, numbers))
print(squared)

#### `map` for parsing — strings in, values out
The genuinely good use of `map`: applying an **existing named function** (like `int`) across incoming text.

In [ ]:
# One CSV line -> list of ints
csv_line = '3,1,4,1,5'
arr = list(map(int, csv_line.split(',')))
print(arr)

# A whole block of rows -> 2-D grid of ints
rows = ['1 2 3', '4 5 6', '7 8 9']
grid = [list(map(int, r.split())) for r in rows]
print(grid)

# Interactive variants:
# arr  = list(map(int, input('Enter comma separated elements: ').split(',')))
# grid = [list(map(int, input().split())) for _ in range(int(input('rows: ')))]

#### `map` with several iterables
Given *n* iterables, the function must take *n* arguments. Like `zip`, it stops at the shortest input.

In [ ]:
list1 = [3, 6, 9]
list2 = [2, 4, 6]
print(list(map(lambda x, y: x + y, list1, list2)))     # element-wise sum

# A conditional transform works too — but compare the comprehension below it
numbers = (1, 2, 3, 4)
print(list(map(lambda x: x + x if x % 2 == 0 else x, numbers)))
print([x + x if x % 2 == 0 else x for x in numbers])

### filter(): Constructs an iterator from elements of an iterable for which a function returns True.
- **Syntax:** `filter(function, iterable)`
    - function : function that tests if each element of a sequence true or not.
    - iterable : It is a iterable which needs to be filtered, it can be sets, lists, tuples, or containers of any iterators.
<img src='./Image/4.2 Image b.png' width=60% height=60%/>

In [ ]:
# filter with a predicate lambda
numbers = (1, 2, 3, 4)
print(list(filter(lambda x: x % 2 == 0, numbers)))

# filter(None, xs) drops FALSY elements — handy for cleaning raw data
raw = ['edge-01', '', 'core-01', None, 'edge-02', '']
print(list(filter(None, raw)))

### reduce(): Used to apply a particular function to all of the elements of iterable.
- This function is defined in “functools” module.
- **Syntax:** `functools.reduce(function, iterable)`
<img src='./Image/4.2 Image c.png' width=60% height=60%/>
- **Working:** 
    - At first step, first two elements of sequence are picked and the result is obtained.
    - Next step is to apply the same function to the previously attained result and the number just succeeding the second element and the result is again stored.
    - This process continues till no more elements are left in the container.
    - The final returned result is returned and printed on console.

### Total via reduce() — and the true cumulative (running) series via itertools.accumulate

In [ ]:
import functools
import itertools

freq = [1, 2, 3, 4, 5]

# reduce() collapses the whole iterable to ONE final value (the total):
total = functools.reduce(lambda x, y: x + y, freq)
print(total)

# For the running (cumulative) series, use itertools.accumulate:
running = list(itertools.accumulate(freq))
print(running)

---

### ⚠️ When to use `map`/`filter`/`reduce` — and when not to

These come from functional languages, and Python supports them — but Python also has
comprehensions, which usually say the same thing more clearly.

| Situation | Prefer |
|---|---|
| `map(lambda x: x*2, xs)` | `[x*2 for x in xs]` — shorter, faster, no `lambda` |
| `map(str, xs)` — an **existing named** function | `map` is genuinely fine here |
| `filter(lambda x: x > 0, xs)` | `[x for x in xs if x > 0]` |
| `reduce(lambda a,b: a+b, xs)` | `sum(xs)` |
| `reduce(lambda a,b: a*b, xs)` | `math.prod(xs)` |
| A genuinely custom fold with an accumulator | `reduce` earns its place |

Guido van Rossum himself argued for removing `reduce` from builtins — which is exactly what
happened in Python 3, where it moved to `functools`.

> **Version note:** in Python 2, `reduce` was a builtin and `map`/`filter` returned **lists**.
> In Python 3, `reduce` requires `from functools import reduce`, and `map`/`filter` return
> **lazy iterators**.

In [ ]:
import timeit

numbers = list(range(1000))

# --- map with a lambda vs a comprehension: same result, different clarity ---
by_map = list(map(lambda x: x * 2, numbers))
by_comp = [x * 2 for x in numbers]
assert by_map == by_comp

t_map = timeit.timeit(lambda: list(map(lambda x: x * 2, numbers)), number=500)
t_comp = timeit.timeit(lambda: [x * 2 for x in numbers], number=500)
print(f"map + lambda   : {t_map * 1000:6.1f} ms")
print(f"comprehension  : {t_comp * 1000:6.1f} ms")

# --- but map with an EXISTING named function is fine, and fast ---
t_named = timeit.timeit(lambda: list(map(str, numbers)), number=500)
t_comp2 = timeit.timeit(lambda: [str(x) for x in numbers], number=500)
print(f"\nmap(str, xs)   : {t_named * 1000:6.1f} ms")
print(f"[str(x) ...]   : {t_comp2 * 1000:6.1f} ms")

# --- filter + lambda vs a comprehension with a condition ---
print("\nfilter :", list(filter(lambda x: x % 2 == 0, range(10))))
print("comp   :", [x for x in range(10) if x % 2 == 0])

# --- reduce, and the builtins that replace it ---
from functools import reduce
import math

print("\nreduce(add) :", reduce(lambda a, b: a + b, [1, 2, 3, 4]))
print("sum()       :", sum([1, 2, 3, 4]), "  <- say what you mean")
print("reduce(mul) :", reduce(lambda a, b: a * b, [1, 2, 3, 4]))
print("math.prod() :", math.prod([1, 2, 3, 4]))

# reduce earns its place for genuinely custom folds
sentences = ["a b", "c", "d e f"]
print("\ncustom fold :", reduce(lambda acc, s: acc + len(s.split()), sentences, 0), "words")

In [ ]:
# The group in action — the functional trio on a small log extract
log = [
    '2026-08-21 10:02:11 ERROR disk /dev/sda1 91%',
    '2026-08-21 10:02:12 INFO heartbeat ok',
    '2026-08-21 10:02:15 ERROR disk /dev/sdb1 88%',
    '2026-08-21 10:02:19 WARN latency 210ms',
]

fields = map(str.split, log)                          # parse: line -> list of fields
errors = filter(lambda f: f[2] == 'ERROR', fields)    # keep ERROR lines only
usages = [int(f[-1].rstrip('%')) for f in errors]     # extract the numbers
print('disk usages on ERROR lines:', usages)
print('worst:', max(usages), '%')

# The same pipeline as one comprehension — usually the clearer spelling:
usages2 = [int(line.split()[-1].rstrip('%'))
           for line in log if line.split()[2] == 'ERROR']
print('same result:', usages2)

> **Note — the next three sections are really OOP material.**
>
> `classmethod()`, `staticmethod()` and `property()` are shown here as *builtin functions*,
> which they technically are. In practice you will always meet them as the decorators
> `@classmethod`, `@staticmethod` and `@property`, applied inside a class body.
>
> They are covered properly in **05 OOPs**, and the decorator machinery that makes them work
> is in **4.4 Function Decorator**. Read them here for completeness; understand them there.

### classmethod()
- When we pass a method as an argument to classmethod(), it converts it into a python class method one that belongs to the class. Then, we call it like we would call any static method in python without an object.
- We can also use the syntactic sugar @classmethod for this.

In [ ]:
class LinearAlgebra:
    def simple_linear(cls):  # a classmethod receives the CLASS as first arg: name it cls
        print("y=mx+c is the equation.")

LinearAlgebra.simple_linear = classmethod(LinearAlgebra.simple_linear)
LinearAlgebra.simple_linear()

In [ ]:
class LinearAlgebra:
    @classmethod
    def simple_linear(cls):  # first parameter is cls, not self
        print("y=mx+c is the equation.")

LinearAlgebra.simple_linear()

### staticmethod()
- staticmethod() creates a static method from a function. 
- A static method is bound to a class rather than to an object. But it can be called on the class or on an object.
- We can also use the syntactic sugar @staticmethod for this.

In [ ]:
class Greet:
    def sayhi():
        print("Hi")
        
Greet.sayhi=staticmethod(Greet.sayhi)
Greet.sayhi()

In [ ]:
class Greet:
    @staticmethod
    def sayhi():
        print("Hi")
        
Greet.sayhi()

### property()
- The function property() returns a property attribute — a managed attribute whose reads (and optionally writes) run getter/setter code.
- We can use the syntactic sugar `@property`. Covered fully in **05 OOPs**.

In [ ]:
class Server:
    def __init__(self, hostname):
        self._hostname = hostname

    @property
    def hostname(self):          # read like an attribute, computed by a method
        return self._hostname.lower()

srv = Server('EDGE-Router-01')
print(srv.hostname)              # no parentheses — the property runs the getter

---

## Quick reference — the builtins in this notebook

| Builtin | Job | Group |
|---|---|---|
| `dir(obj)` | List attribute names | Introspection |
| `help(obj)` | Interactive docs (reads `__doc__`) | Introspection |
| `isinstance(x, T)` | Inheritance-aware type test | Introspection |
| `callable(x)` | Can it be called? | Introspection |
| `getattr` / `setattr` / `hasattr` / `delattr` | Attribute access by name | Introspection |
| `vars(obj)` | The object's `__dict__` | Introspection |
| `type(x)` | The exact class | Introspection |
| `object()` | Featureless base / unique sentinel | Introspection |
| `enumerate(it, start=0)` | Lazy `(index, item)` pairs | Iteration |
| `zip(*its)` | Parallel iteration (`strict=True` on 3.10+) | Iteration |
| `reversed(seq)` | Lazy backwards iterator | Iteration |
| `sorted(it, key=..., reverse=...)` | New sorted list | Iteration |
| `len` / `sum` / `min` / `max` | Count / total / extremes | Aggregation |
| `any` / `all` | Collective truth, short-circuiting | Aggregation |
| `abs` / `divmod` / `pow` / `round` | Numeric helpers | Aggregation |
| `int` / `float` / `str` / `bool` | Scalar conversion | Conversion |
| `list` / `tuple` / `set` / `dict` | Container construction / conversion | Conversion |
| `map(f, it)` | Apply `f` to every element, lazily | Functional |
| `filter(pred, it)` | Keep approved elements, lazily | Functional |
| `functools.reduce(f, it)` | Fold to one value (needs the import!) | Functional |
| `classmethod` / `staticmethod` / `property` | Method kinds — see **05 OOPs** | OOP |

---

## Common Mistakes & Pitfalls

1. **Forgetting `map`/`filter`/`zip`/`enumerate` return iterators, not lists.** Printing one shows `<map object at 0x...>`. Wrap in `list()` — and remember it can only be consumed once.
2. **Using `reduce` without importing it.** In Python 3 it lives in `functools`; it was a builtin in Python 2.
3. **Reaching for `map(lambda ...)` where a comprehension is clearer.** `map(lambda x: x*2, xs)` is longer *and* slower than `[x*2 for x in xs]`.
4. **Using `reduce` for a sum, max or min.** `sum()`, `max()` and `min()` already exist and say what they mean.
5. **Calling `sorted(key=len())` instead of `key=len`.** `key` takes the function itself, uncalled — no parentheses.
6. **Confusing `sort()` and `sorted()`** — `sort()` returns `None` (see **2.4**).
7. **Using `isinstance(x, int)` to reject `bool`.** `bool` subclasses `int`, so `isinstance(True, int)` is `True`.
8. **Using `eval()` to look up an attribute by name.** That's what `getattr()` is for.
9. **Relying on `zip` to check lengths.** It silently stops at the shortest input — pass `strict=True` (3.10+) when the lengths must match.
10. **`bool('False')` is `True`.** Any non-empty string is truthy — parse config strings explicitly.

## Best Practices

- Prefer a **comprehension** over `map`/`filter` with a `lambda`; use `map(str, xs)` only when the function already exists and is named.
- Use `sum()`, `max()`, `min()`, `any()`, `all()` instead of hand-rolled `reduce` calls.
- Pass `key=` to `sorted`/`max`/`min` rather than transforming the data first.
- Use `operator.itemgetter` / `attrgetter` instead of `lambda x: x[1]` — faster and clearer.
- Use `isinstance()` rather than comparing `type(x) == SomeType`; it respects inheritance.
- Keep `lambda` to a single short expression. If it needs a name, use `def`.
- Use `getattr(obj, name, default)` for dynamic attribute access — never `eval`.
- Pass `strict=True` to `zip` when the inputs must be the same length.
- Use `filter(None, xs)` to drop falsy elements from raw data in one step.

## Practice Exercises

Try these before moving on.

1. Rewrite three `map`/`filter` examples from this notebook as comprehensions. Which read better?
2. Sort a list of `(name, age)` tuples by age, then by name, using `itemgetter`.
3. Use `max()` with `key=` to find the longest word in a sentence.
4. Use `functools.reduce` to compute a factorial, then do it with `math.prod`. Which is clearer?
5. Write a function that takes an object and a list of attribute names and returns their values using `getattr`.
6. Show why `isinstance(True, int)` is `True`, and write a check that accepts `int` but rejects `bool`.
7. Use `zip`, `map` and `sum` together to compute a dot product in one line.
8. Write `common(l1, l2)` returning the values present in both lists — first with loops, then with set intersection. Which is clearer, and which is faster?
9. Use `divmod` twice to format `9876` seconds as `h:mm:ss`.